# 뮤직비디오 시각(영상) 특징 추출

K-pop 뮤직비디오 영상에서 시각적 특징을 나타내는 지표를 추출하는 코드입니다. 유튜브 영상을 다운로드해 프레임을 샘플링한 뒤 `OpenCV`로 밝기, 모션, 색상 등의 지표를 계산합니다.

- **avg_brightness** — 평균 밝기
- **avg_motion** — 프레임 간 변화량(움직임 정도)
- **avg_r_value / avg_g_value / avg_b_value** — 평균 RGB 값

**구성**
1. 환경 설정 — 필요한 패키지 설치
2. 메인 시각 특징 추출 — 전체 곡 목록을 대상으로 영상 다운로드 및 프레임 분석
3. 실패 항목 재추출 — 추출에 실패한 곡만 골라내 다시 시도

**참고사항**
- 이 코드는 Google Colab 환경(`/content/...` 경로, `google.colab.files`)을 기준으로 작성되었습니다.
- 원본 곡 목록과 추출 결과 파일은 저작권이 있는 곡 정보를 포함하고 있어 이 저장소에는 포함하지 않았습니다.


## 1. 환경 설정

영상 다운로드/분석에 필요한 패키지(`yt-dlp`, `opencv-python`, `pandas`, `numpy`)를 설치합니다.


In [ ]:
!pip install yt-dlp opencv-python pandas numpy

## 2. 메인 시각 특징 추출

`VideoFeatureExtractor` 클래스로 곡 목록의 각 유튜브 URL에서 영상을 다운로드하고, 일정 간격으로 프레임을 샘플링해 밝기·모션·RGB 값을 계산한 뒤 원본 데이터와 병합해 CSV로 저장합니다.


In [ ]:
"""
뮤직비디오 시각적 특성 추출 (OpenCV)
실행 전 준비: pip install opencv-python yt-dlp pandas numpy
"""

import cv2
import numpy as np
import pandas as pd
import yt_dlp
import os
import tempfile
from urllib.parse import urlparse, parse_qs
import time

class VideoFeatureExtractor:
    def __init__(self, sample_fps=1):
        """
        Parameters:
        -----------
        sample_fps : int
            샘플링 프레임 레이트 (1 = 1초당 1프레임)
            높을수록 정확하지만 느림
        """
        self.sample_fps = sample_fps
        self.temp_dir = tempfile.mkdtemp()

    def extract_video_id(self, url):
        """YouTube URL에서 비디오 ID 추출"""
        try:
            parsed_url = urlparse(url)
            if 'youtube.com' in parsed_url.netloc:
                query_params = parse_qs(parsed_url.query)
                return query_params.get('v', [None])[0]
            elif 'youtu.be' in parsed_url.netloc:
                return parsed_url.path[1:]
        except:
            return None
        return None

    def download_video(self, url, output_path):
        """
        YouTube 비디오 다운로드

        Returns:
        --------
        str or None: 다운로드된 파일 경로
        """
        ydl_opts = {
            'format': 'worst[ext=mp4]',  # 가장 낮은 화질 (빠름)
            'outtmpl': output_path,
            'quiet': True,
            'no_warnings': True,
        }

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            return output_path
        except Exception as e:
            print(f"  ⚠️ 다운로드 실패: {e}")
            return None

    def calculate_brightness(self, frame):
        """평균 밝기 계산 (0-255)"""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return np.mean(gray)

    def calculate_motion(self, frame1, frame2):
        """
        프레임 간 모션 계산 (Frame Difference)

        Returns:
        --------
        float: 모션 정도 (0-255, 높을수록 움직임 많음)
        """
        gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Frame difference
        diff = cv2.absdiff(gray1, gray2)

        return np.mean(diff)

    def calculate_rgb_values(self, frame):
        """
        평균 RGB 값 계산

        Returns:
        --------
        tuple: (avg_r, avg_g, avg_b)
        """
        # OpenCV는 BGR 순서
        b, g, r = cv2.split(frame)

        return (
            np.mean(r),
            np.mean(g),
            np.mean(b)
        )

    def analyze_video(self, video_path):
        """
        비디오 분석

        Returns:
        --------
        dict or None: {
            'avg_brightness': float,
            'avg_motion': float,
            'avg_r_value': float,
            'avg_g_value': float,
            'avg_b_value': float,
            'frames_analyzed': int,
            'video_duration': float
        }
        """
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"  ⚠️ 비디오를 열 수 없음: {video_path}")
            return None

        # 비디오 정보
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0

        # 샘플링 간격 계산
        frame_interval = int(fps / self.sample_fps) if fps > 0 else 1

        brightness_values = []
        motion_values = []
        r_values = []
        g_values = []
        b_values = []

        prev_frame = None
        frame_count = 0
        analyzed_count = 0

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            # 샘플링: 지정된 간격마다만 분석
            if frame_count % frame_interval == 0:
                # Brightness
                brightness = self.calculate_brightness(frame)
                brightness_values.append(brightness)

                # Motion (이전 프레임과 비교)
                if prev_frame is not None:
                    motion = self.calculate_motion(prev_frame, frame)
                    motion_values.append(motion)

                # RGB
                r, g, b = self.calculate_rgb_values(frame)
                r_values.append(r)
                g_values.append(g)
                b_values.append(b)

                prev_frame = frame.copy()
                analyzed_count += 1

            frame_count += 1

        cap.release()

        # 평균 계산
        result = {
            'avg_brightness': np.mean(brightness_values) if brightness_values else None,
            'avg_motion': np.mean(motion_values) if motion_values else None,
            'avg_r_value': np.mean(r_values) if r_values else None,
            'avg_g_value': np.mean(g_values) if g_values else None,
            'avg_b_value': np.mean(b_values) if b_values else None,
            'frames_analyzed': analyzed_count,
            'video_duration': duration,
            'video_fps': fps
        }

        return result

    def process_url(self, url, video_id=None):
        """
        YouTube URL에서 비디오 다운로드 및 분석

        Returns:
        --------
        dict or None: 분석 결과
        """
        if video_id is None:
            video_id = self.extract_video_id(url)

        if not video_id:
            print("  ⚠️ 비디오 ID를 추출할 수 없음")
            return None

        # 임시 파일 경로
        temp_video_path = os.path.join(self.temp_dir, f"{video_id}.mp4")

        try:
            # 1. 비디오 다운로드
            print(f"  📥 다운로드 중...")
            downloaded_path = self.download_video(url, temp_video_path)

            if not downloaded_path or not os.path.exists(downloaded_path):
                return None

            # 2. 비디오 분석
            print(f"  🎬 분석 중...")
            result = self.analyze_video(downloaded_path)

            # 3. 임시 파일 삭제
            if os.path.exists(downloaded_path):
                os.remove(downloaded_path)

            return result

        except Exception as e:
            print(f"  ⚠️ 처리 오류: {e}")
            # 오류 발생 시에도 임시 파일 삭제
            if os.path.exists(temp_video_path):
                os.remove(temp_video_path)
            return None

    def process_dataframe(self, df, url_col='url', video_id_col='video_id'):
        """
        DataFrame의 모든 비디오 처리

        Parameters:
        -----------
        df : pd.DataFrame
            처리할 데이터프레임
        url_col : str
            URL 컬럼명
        video_id_col : str
            비디오 ID 컬럼명 (있으면)

        Returns:
        --------
        pd.DataFrame: 시각적 특성이 추가된 데이터프레임
        """
        total = len(df)
        results = []

        print(f"\n{'='*60}")
        print(f"비디오 시각적 특성 추출 시작 ({total}개)")
        print(f"샘플링: {self.sample_fps} FPS")
        print(f"{'='*60}\n")

        for idx, row in df.iterrows():
            url = row[url_col]
            video_id = row.get(video_id_col) if video_id_col in df.columns else None

            print(f"\n[{idx + 1}/{total}] {url}")

            result = self.process_url(url, video_id)

            if result:
                print(f"  ✓ 완료 - 밝기: {result['avg_brightness']:.1f}, "
                      f"모션: {result['avg_motion']:.1f}, "
                      f"프레임: {result['frames_analyzed']}")
            else:
                result = {}

            results.append(result)

            # 진행률 표시
            if (idx + 1) % 10 == 0:
                success = sum(1 for r in results if r)
                print(f"\n📊 진행률: {idx + 1}/{total} ({(idx + 1) / total * 100:.1f}%)")
                print(f"   성공: {success}, 실패: {idx + 1 - success}")

        # 결과 DataFrame 생성
        results_df = pd.DataFrame(results)

        # 원본과 병합
        final_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

        # 통계
        success = sum(1 for r in results if r)
        print(f"\n{'='*60}")
        print(f"처리 완료")
        print(f"{'='*60}")
        print(f"✓ 성공: {success}/{total} ({success / total * 100:.1f}%)")
        print(f"✗ 실패: {total - success}/{total}")

        return final_df

    def cleanup(self):
        """임시 디렉토리 정리"""
        import shutil
        if os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)


def main():
    """메인 실행 함수"""

    # ========================================
    # 📁 파일 경로 설정
    # ========================================
    INPUT_CSV = "/content/bottom_500_balanced.csv"
    OUTPUT_CSV = "with_visual_features_bottom.csv"

    # ========================================
    # ⚙️ 샘플링 설정
    # ========================================
    # sample_fps = 1: 1초당 1프레임 (빠름, 덜 정확)
    # sample_fps = 2: 1초당 2프레임 (보통)
    # sample_fps = 5: 1초당 5프레임 (느림, 정확)
    SAMPLE_FPS = 2

    print(f"\n{'='*60}")
    print("뮤직비디오 시각적 특성 추출")
    print(f"{'='*60}\n")

    # CSV 읽기
    print("📂 CSV 파일 읽기...")
    df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
    print(f"✓ 총 {len(df):,}개 레코드\n")


    # Extractor 생성
    extractor = VideoFeatureExtractor(sample_fps=SAMPLE_FPS)

    try:
        # 비디오 처리
        result_df = extractor.process_dataframe(df)

        # 저장
        print(f"\n💾 결과 저장: {OUTPUT_CSV}")
        result_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print("✓ 저장 완료!")

        print(f"\n{'='*60}")
        print("🎉 모든 작업 완료!")
        print(f"{'='*60}")

    finally:
        # 정리
        extractor.cleanup()


if __name__ == "__main__":
    main()

## 3. 실패 항목 재추출

기존 처리 결과에서 다운로드/분석에 실패한(`status`가 성공이 아닌) 행만 골라내 봇 차단 회피용 설정(User-Agent, 재시도 횟수, 딜레이)을 강화해서 다시 다운로드·분석합니다. 성공분과 재시도분을 합쳐 최종 결과를 저장합니다.


In [ ]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
import yt_dlp
from tqdm import tqdm
import tempfile
import os
import time
import glob
from google.colab import files

def download_youtube_video(url, output_dir, video_id, max_retries=3):
    """YouTube 비디오를 다운로드"""

    output_template = os.path.join(output_dir, f'{video_id}.%(ext)s')

    ydl_opts = {
        'format': 'worst[ext=mp4]/worst',
        'outtmpl': output_template,
        'quiet': True,
        'no_warnings': True,
        'socket_timeout': 30,
        'retries': 10,
        'fragment_retries': 10,
        'nocheckcertificate': True,
        'merge_output_format': 'mp4',
        'extractor_args': {
            'youtube': {
                'player_client': ['android', 'web'],
                'skip': ['hls', 'dash']
            }
        },
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-us,en;q=0.5',
            'Sec-Fetch-Mode': 'navigate',
        },
    }

    for attempt in range(max_retries):
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)

                possible_files = glob.glob(os.path.join(output_dir, f'{video_id}.*'))

                if possible_files:
                    downloaded_file = possible_files[0]
                    file_size = os.path.getsize(downloaded_file)

                    if file_size > 1000:
                        return downloaded_file

                time.sleep(2)

        except Exception as e:
            error_msg = str(e)

            if any(keyword in error_msg for keyword in ['Sign in', 'age', 'Private', 'unavailable', 'bot']):
                return None

            if attempt < max_retries - 1:
                time.sleep(5)

    return None

def analyze_video(video_path, sample_rate=30):
    """OpenCV를 사용하여 비디오 분석"""
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    brightness_values = []
    r_values = []
    g_values = []
    b_values = []
    motion_values = []

    prev_frame = None
    frame_count = 0
    analyzed_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % sample_rate == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            brightness = np.mean(gray)
            brightness_values.append(brightness)

            b, g, r = cv2.split(frame)
            r_values.append(np.mean(r))
            g_values.append(np.mean(g))
            b_values.append(np.mean(b))

            if prev_frame is not None:
                diff = cv2.absdiff(gray, prev_frame)
                motion = np.mean(diff)
                motion_values.append(motion)

            prev_frame = gray.copy()
            analyzed_count += 1

        frame_count += 1

    cap.release()

    result = {
        'avg_brightness': round(np.mean(brightness_values), 2) if brightness_values else 0,
        'avg_motion': round(np.mean(motion_values), 2) if motion_values else 0,
        'avg_r_value': round(np.mean(r_values), 2) if r_values else 0,
        'avg_g_value': round(np.mean(g_values), 2) if g_values else 0,
        'avg_b_value': round(np.mean(b_values), 2) if b_values else 0,
        'frames_analyzed': analyzed_count,
        'video_duration': round(duration, 2),
        'video_fps': round(fps, 2)
    }

    return result

def retry_failed_videos(existing_csv, original_csv, url_column='url',
                       output_csv='video_analysis_retry.csv',
                       sample_rate=30, delay_between_downloads=7):
    """실패한 비디오만 재시도"""

    # 기존 결과 읽기
    existing_df = pd.read_csv(existing_csv)
    original_df = pd.read_csv(original_csv)

    # 실패한 것들 찾기 (download_failed, analysis_failed, error)
    failed_statuses = ['download_failed', 'analysis_failed', 'error']

    # 성공한 것들의 인덱스
    if 'status' in existing_df.columns:
        success_indices = existing_df[existing_df['status'] == 'success'].index.tolist()

        # 전체에서 성공한 것 제외
        failed_indices = [i for i in range(len(original_df)) if i not in success_indices]
        failed_df = original_df.iloc[failed_indices]
    else:
        # status 컬럼이 없으면 처리되지 않은 것으로 간주
        failed_df = original_df.iloc[len(existing_df):]

    print("="*70)
    print("🔄 실패한 비디오 재시도")
    print("="*70)
    print(f"✅ 이미 성공: {len(success_indices)}개")
    print(f"❌ 재시도 필요: {len(failed_df)}개")
    print(f"⏱️  딜레이: {delay_between_downloads}초 (봇 차단 회피)")
    print("="*70)

    # yt-dlp 업데이트
    print("\n📦 yt-dlp 업데이트 중...")
    os.system('pip install -U yt-dlp --quiet')
    print("✓ 업데이트 완료\n")

    results = []
    temp_dir = tempfile.mkdtemp()
    start_time = time.time()

    for idx, (orig_idx, row) in enumerate(tqdm(failed_df.iterrows(), total=len(failed_df), desc="🔄 재시도")):
        url = row[url_column]
        video_id = f'retry_{orig_idx}'

        print(f"\n[{idx+1}/{len(failed_df)}] 원본 #{orig_idx} 재시도", end=" ")

        try:
            downloaded_file = download_youtube_video(url, temp_dir, video_id)

            if downloaded_file:
                file_size = os.path.getsize(downloaded_file)
                print(f"다운({file_size/(1024*1024):.1f}MB)", end=" → ")

                features = analyze_video(downloaded_file, sample_rate=sample_rate)

                if features:
                    result = {
                        **row.to_dict(),
                        **features,
                        'status': 'success',
                        'file_size_mb': round(file_size/(1024*1024), 2),
                        'original_index': orig_idx
                    }
                    print(f"분석완료({features['video_duration']:.0f}초) ✓")
                else:
                    result = {**row.to_dict(), 'status': 'analysis_failed', 'original_index': orig_idx}
                    print("분석실패 ✗")

                os.remove(downloaded_file)
            else:
                result = {**row.to_dict(), 'status': 'download_failed', 'original_index': orig_idx}
                print("다운실패 ✗")

            time.sleep(delay_between_downloads)

        except Exception as e:
            result = {**row.to_dict(), 'status': 'error', 'original_index': orig_idx}
            print(f"에러 ✗")

        results.append(result)

        # 중간 저장 (10개마다)
        if len(results) % 10 == 0:
            retry_df = pd.DataFrame(results)
            retry_df.to_csv(output_csv, index=False)

            success = sum(1 for r in results if r.get('status') == 'success')
            print(f"\n💾 저장: {len(results)}개 처리, 성공 {success}개 ({success/len(results)*100:.1f}%)\n")

    # 최종 저장
    retry_df = pd.DataFrame(results)
    retry_df.to_csv(output_csv, index=False)

    # 기존 성공한 것과 합치기
    success_df = existing_df[existing_df['status'] == 'success'].copy()
    combined_df = pd.concat([success_df, retry_df], ignore_index=True)

    # original_index로 정렬
    if 'original_index' in combined_df.columns:
        combined_df = combined_df.sort_values('original_index').reset_index(drop=True)

    combined_file = output_csv.replace('.csv', '_combined.csv')
    combined_df.to_csv(combined_file, index=False)

    # 통계
    print("\n" + "="*70)
    print("✅ 재시도 완료!")
    print("="*70)
    print(f"\n📊 재시도 결과:")
    print(retry_df['status'].value_counts())
    print(f"\n📊 전체 통계 (기존 + 재시도):")
    print(combined_df['status'].value_counts())
    print(f"\n💾 저장 파일:")
    print(f"   재시도 결과: {output_csv}")
    print(f"   전체 결과: {combined_file}")

    # 다운로드
    print("\n📥 로컬 PC로 다운로드 중...")
    try:
        files.download(output_csv)
        files.download(combined_file)
        print("✓ 다운로드 완료!")
    except:
        print("⚠️ 파일 탭에서 수동 다운로드하세요")

    # 정리
    try:
        for file in glob.glob(os.path.join(temp_dir, '*')):
            os.remove(file)
        os.rmdir(temp_dir)
    except:
        pass

    return combined_df

# 실행
if __name__ == "__main__":
    # 가장 최근 백업 파일 찾기
    backup_files = sorted(glob.glob('/content/시각정보_미추출_행만_backup.csv'))

    if backup_files:
        latest_backup = backup_files[-1]
        print(f"📂 백업 파일 발견: {latest_backup}")

        result = retry_failed_videos(
            existing_csv=latest_backup,  # 백업 파일
            original_csv='/content/시각정보_미추출_행만.csv',  # 원본 파일
            url_column='url',
            output_csv='kpop_video_analysis_retry.csv',
            sample_rate=30,
            delay_between_downloads=7  # 7초로 늘림 (봇 차단 회피)
        )
    else:
        print("❌ 백업 파일을 찾을 수 없습니다!")